# 07 — Wind models in MechaTree

Wind enters the simulator in three distinct places, and an outside
reader can't tell which knob does what without a map. This notebook
is that map. By the end you will know:

1. **Mechanics**: how a wind vector becomes a per-branch stress, and
   why the safety-factor genome uses a 'sensing sweep' over several
   directions.
2. **Storm wind statistics**: amplitude and angle distributions are
   tunable via SymPy CDFs in the YAML `wind:` block (Step 25). The
   default still reproduces the Fortran code byte-identically.
3. **Canopy-aware wind**: the native 3-D momentum-wind CFD model, with
   rotation to face the storm and a per-branch screened wind field
   feeding both sensing and pruning.
4. **Step 24 in pictures**: the wind ↔ pruning fixed-point loop,
   replayed pass-by-pass on a small forest hit with a single big
   storm. Pruned branches are drawn in black so you can watch the
   canopy collapse iteration by iteration.
5. **A lab section**: swap the angle distribution from uniform to a
   prevailing-wind von Mises, re-run, and see the canopy adapt.

In [ ]:
import math
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

import mechatree as mt

mt.figstyle.apply()
rng = np.random.default_rng(0)

# Load the shared tree config from examples/forest.yaml so edits to the
# YAML (angles theta1/theta2/gamma1/gamma2, leaf_surface, cauchy, etc.)
# flow through this notebook. The warmup cells below use TREE_CFG
# instead of bare mt.TreeConfig() defaults.
TREE_CFG = mt.load_config(Path("..") / "examples" / "forest.yaml").tree
print(
    f"TREE_CFG angles: theta1={TREE_CFG.theta1:.3f}, "
    f"theta2={TREE_CFG.theta2:.3f}, "
    f"gamma1={TREE_CFG.gamma1:.3f}, gamma2={TREE_CFG.gamma2:.3f}"
)

## 1. From a wind vector to a per-branch stress

Every branch is a tapered cylinder of length $L$ and diameter $D$ with
a unit axis $\hat{t}$. Given a wind vector $\mathbf{U}$ (units arbitrary;
MechaTree is unit-free), the drag on the branch is proportional to its
frontal area, which depends on the angle $I$ between $\hat{t}$ and
$\mathbf{U}$:

$$F = \tfrac{1}{2} \, |\mathbf{U}|^2 \, D \, L \, \sin^2 I.$$

(The $\tfrac{1}{2}$ is the $\tfrac{1}{2}\rho U^2$ dynamic-pressure factor;
$\rho = C_D = 1$. The original Fortran dropped it and folded it into the
Cauchy calibration — Step 26 restored it and doubled the default
`cauchy` so the default model still reproduces the reference exactly.)

That force is integrated up the tree (children pass their force and
moment to their parent) and the resulting bending stress at each
branch is

$$\sigma = \frac{16}{\pi} \, c_y \, \frac{|\hat{t} \times \mathbf{M}|}{D^3}.$$

This is exactly what [`mechatree._core.mechanics.cpp`](../src/mechatree/_core/mechanics.cpp)
implements.

**Two distinct uses of the stress.** *Sensing* drives growth: the
safety-factor genome reads each branch's worst-case stress across a
sweep of wind directions and sets the diameter so that
$\sigma_\mathrm{max} \le \sigma_{\mathrm{break}} / k$ for some safety
factor $k$. *Pruning* uses the per-generation storm wind: each branch's
instantaneous stress is plugged into a Weibull failure law
(`pruning.cpp`) and the branch falls with probability $p_\mathrm{fail}$.

For the `momentum` model, both sensing and pruning use the same
**screened per-branch wind** (Step 26c): sensing sweeps
`n_sensing_angles` directions drawn from `angle_cdf`, each a CFD solve
at unit inlet, and keeps the per-branch max stress. The `default`
(canopy-blind) model keeps the legacy hard-wired four-angle sweep
$\theta \in \{\pi/4, \pi/2, 3\pi/4, \pi\}$ at unit wind.

## 2. Storm wind statistics — tunable distributions

The Fortran reference sampled the storm angle deterministically as
$\theta = g$ rad (rotating by 1 rad per generation) and the amplitude
from a shifted exponential $a = 0.835 - \log(U) / 6$, where $U$ is
uniform on $(0, 1]$. That gives a typical amplitude $\approx 1.0$ with
rare gusts reaching $\sim 3$.

Step 25 lets you swap either with a SymPy CDF in YAML:

```yaml
wind:
  model: default
  amplitude_cdf: "1 - exp(-6 * (a - 0.835))"   # default, reproduces Fortran
  angle_cdf: "theta / (2*pi)"                  # uniform on [0, 2*pi)
```

Leaving a field unset keeps the legacy behaviour byte-identically, so
existing simulations don't drift.

Let's compare the legacy amplitude distribution against a heavier-tail
alternative (Rayleigh, mean $\sqrt{\pi/2} \approx 1.25$):

In [ ]:
default_amp = mt.default_amplitude_sampler()
rayleigh_amp = mt.Distribution(
    cdf_expr="1 - exp(-x**2 / 2)", var_name="x", support=(0.0, math.inf)
).sampler()

default_samples = default_amp(rng, 10_000)
rayleigh_samples = rayleigh_amp(rng, 10_000)

fig = mt.figstyle.figure(size="full", aspect=9 / 4)
fig.add_trace(
    go.Histogram(
        x=default_samples,
        name="Fortran default",
        nbinsx=80,
        histnorm="probability density",
        opacity=0.6,
        marker_color=mt.figstyle.COLORS["blue"],
    )
)
fig.add_trace(
    go.Histogram(
        x=rayleigh_samples,
        name="Rayleigh",
        nbinsx=80,
        histnorm="probability density",
        opacity=0.6,
        marker_color=mt.figstyle.COLORS["red"],
    )
)
fig.update_layout(barmode="overlay", xaxis_title="storm amplitude", yaxis_title="density")
fig.show()
print(
    f"Fortran:  mean={default_samples.mean():.3f}, "
    f"99th pct={np.percentile(default_samples, 99):.3f}"
)
print(
    f"Rayleigh: mean={rayleigh_samples.mean():.3f}, "
    f"99th pct={np.percentile(rayleigh_samples, 99):.3f}"
)

Same machinery for the angle distribution. Default is uniform on
$[0, 2\pi)$ (the canopy is hit isotropically); a prevailing-wind setup
would use e.g. a von Mises CDF centred on $\theta = 0$.

## 3. Canopy-aware wind — the momentum model

Without a canopy model, every branch in a forest feels the same
free-stream wind. That's wrong: dense canopies absorb momentum, so the
branches downstream feel a thinner wind than the ones at the leading
edge. MechaTree's canopy-aware path is the native 3-D **momentum-wind**
CFD solver in
[`mechatree.wind.momentum_wind`](../src/mechatree/wind/momentum_wind.py)
(`model: momentum`) — pure NumPy, no external dependency.

Key feature: the model **rotates the canopy to face the storm**, the
same way [`mechatree.light.interception`](../src/mechatree/light/interception.py)
rotates leaves to face the sun. A storm from the north and a storm from
the east give genuinely different wakes — each branch's projected area
$D \, L \, \sin I$ and local screened wind are recomputed for the right
direction.

The cell below runs the bridge for three storm directions and reads
back the screened **canopy-mean** — a single scalar that the Step-24
fixed-point loop uses only as a convergence thermometer (the wind that
actually drives pruning is the *per-branch* field, §3.5). The magnitude
is direction-independent and the recovered angle matches the input,
confirming the rotation round-trips.

(Step 26 removed the legacy `native` bulk-thinning and `dendroflow`
bridges — both collapsed the 3-D field to a single scalar, equivalent
to a constant wind; the per-branch momentum model supersedes them.)

In [ ]:
from mechatree.wind.momentum_wind import MomentumWindBridge

# Build a tiny forest first. TREE_CFG comes from examples/forest.yaml
# (see cell 1) so the YAML angles theta1/theta2/gamma1/gamma2 are
# the ones actually used here.
cfg = mt.Config(
    tree=TREE_CFG,
    forest=mt.ForestConfig(size=15.0, n_trees_init=8, n_trees_max=80),
    n_generations=30,
)
f = mt.Forest(cfg, seed=0)
for g in range(30):
    f.step(g)
print(
    f"warmup forest: {len(f.trees)} trees, "
    f"{sum(t.get_number_of_branches() for t in f.trees)} branches"
)

# Run the momentum bridge for three storm directions. A fresh bridge per
# direction whose angle_sampler returns the fixed angle; the bridge rotates
# the canopy to face it, solves, and reports the screened canopy-mean back
# in the world frame.
for deg in (0.0, 45.0, 90.0):
    th = math.radians(deg)
    bridge = MomentumWindBridge(
        grid_size=2.0,
        pad_x=8.0,
        U_uniform=3.0,
        angle_sampler=lambda r, n, th=th: np.array([th]),
    )
    cm = bridge(0, rng, f)  # screened canopy-mean, world frame
    mag = math.hypot(cm[0], cm[1])
    recovered = math.degrees(math.atan2(cm[1], cm[0]))
    print(
        f"θ={deg:>4.0f}°  canopy_mean=({cm[0]:+.3f}, {cm[1]:+.3f})  "
        f"|U|={mag:.3f}  recovered θ={recovered:+.2f}°"
    )

### 3.5 Per-branch wind & wake plane

Cell 6 read back the **canopy-mean** — a single scalar. That's only the
convergence thermometer; the momentum model's real output is the
**per-branch local wind**, which is what sensing and pruning actually
use (Step 26c). Two trees side-by-side at the same height see *different*
wind once one is in the other's wake — exactly what a canopy-mean throws
away.

The solver
([`mechatree.wind.momentum_wind`](../src/mechatree/wind/momentum_wind.py))
marches a structured grid column-by-column in x, computes per-cell drag
from the cylinders inside (Nat Commun 2017 kernel
$F_D = \tfrac{1}{2} C_D D L \sin^3 i \cdot U^2$), solves the per-cell
momentum balance ($U_\mathrm{out} = \tfrac{1}{2} U_\mathrm{in}[1 + \sqrt{1 - 4 F_D / (H^2 U_\mathrm{in}^2)}]$),
and smooths cross-stream with a single dimensionless diffusion
coefficient `nu_diff`. The output is a 3-D field $U_\mathrm{out}(x, y, z)$
so the streamwise wake shows up. Derivation in
[docs/momentum_wind_derivation.md](../docs/momentum_wind_derivation.md).

Four parameters that matter:

- `grid_size` — grid cell size (uniform x/y/z). Must satisfy
  $\text{grid\_size} \ge \max L$ so each branch fits in one cell
  (centroid binning, no subdivision). Default 2 is the
  $O(\text{branches}) \approx O(\text{cells})$ cost knee that still
  resolves the vertical + intra-crown gradient.
- `nu_diff` — cross-stream diffusion coefficient. **The only
  turbulence knob.** Larger ⇒ wider wake; `nu_diff = 0` ⇒ sharp
  wake with no lateral mixing. Default `0.03`.
- `U_infty` — inflow profile. Here we use a **uniform** free stream
  $U_\mathrm{in} = K$, independent of height, set to the
  $\approx$1-in-100-yr storm amplitude: the same long-tail formula the
  default wind samples ($a = 0.835 - \log U / 6$) evaluated at
  exceedance $U = 1/100$, so $K \approx 1.60$. (Pass `(ua, z0, kappa)`
  to the bridge instead — or set `momentum_U_uniform` in the YAML — to
  choose between a log-law and this uniform inflow.)
- `C_D` — branch drag coefficient.

Wall-clock: a few ms on the warmup forest (~5 k branches). Single-pass,
pure NumPy, no external dependency.

The figure: each branch's segment coloured by $U_\mathrm{branch} /
U_\infty$ — its local screened wind, normalised by the free stream $K$.
A semi-transparent yz plane is parked downwind showing
$U_\mathrm{out}(y, z) / U_\infty$ in the last x-cell, so the wake reads
as a 2-D pattern that *does* vary in y.

In [ ]:
# 3-D wind field via the native momentum-wind kernel. Single-pass:
# run once on the warmup forest, look up each branch's wind in the
# cell containing its midpoint, render a downwind yz plane coloured
# by U_out / U_infty.
import time

import plotly.colors as pcolors

from mechatree.wind._momentum_wind_kernel import compute_momentum_wind

# Pool the warmup forest's per-branch geometry (f comes from cell 6).
starts, axes, Ds, Ls = [], [], [], []
for t in f.trees:
    s, a, d, ell = t.get_branch_data_batch()
    starts.append(s)
    axes.append(a)
    Ds.append(d)
    Ls.append(ell)
start = np.concatenate(starts)
axis = np.concatenate(axes)
D = np.concatenate(Ds)
L = np.concatenate(Ls)

# Structured grid covering the canopy + downstream wake room.
H_cell = 2.0
x_lo, x_hi = start[:, 0].min() - 3.0, start[:, 0].max() + 12.0
y_lo, y_hi = start[:, 1].min() - 2.0, start[:, 1].max() + 2.0
z_hi = (start[:, 2] + L * axis[:, 2]).max() + 3.0
cell_bounds_x = np.arange(x_lo, x_hi + H_cell, H_cell)
cell_bounds_y = np.arange(y_lo, y_hi + H_cell, H_cell)
cell_bounds_z = np.arange(0.0, z_hi + H_cell, H_cell)
z_centers_grid = 0.5 * (cell_bounds_z[:-1] + cell_bounds_z[1:])

# Uniform inflow: U_in = K independent of height z. K is the
# ~1-in-100-yr storm amplitude, from the same long-tail formula the
# default wind samples (a = 0.835 - log(U)/6), at exceedance U = 1/100.
K = 0.835 - np.log(1 / 100) / 6  # ≈ 1.60
U_infty = np.full(z_centers_grid.size, K)

t0 = time.time()
res = compute_momentum_wind(
    start,
    axis,
    D,
    L,
    cell_bounds_x=cell_bounds_x,
    cell_bounds_y=cell_bounds_y,
    cell_bounds_z=cell_bounds_z,
    grid_size=H_cell,
    U_infty=U_infty,
    C_D=1.0,
    nu_diff=0.03,
    wind_direction=(1.0, 0.0),
)
print(
    f"grid: Nx={cell_bounds_x.size - 1}, Ny={cell_bounds_y.size - 1}, "
    f"Nz={cell_bounds_z.size - 1}, cell size grid_size={H_cell}"
)
print(f"momentum-wind wall-clock: {time.time() - t0:.2f}s")
U_out = res.U_out
print(
    f"U_infty (uniform): {U_infty.min():.2f} … {U_infty.max():.2f}; "
    f"U_out: {U_out.min():.2f} … {U_out.max():.2f}"
)
# Per-branch results come directly from the kernel.
U_inf_branch = U_infty[
    np.clip(
        np.searchsorted(cell_bounds_z, start[:, 2] + 0.5 * L * axis[:, 2], side="right") - 1,
        0,
        cell_bounds_z.size - 2,
    )
]
ratio = res.U_branch / np.maximum(U_inf_branch, 1e-6)
print(
    f"per-branch wind: {res.U_branch.min():.2f} … {res.U_branch.max():.2f} "
    f"(ratio {ratio.min():.2f} … {ratio.max():.2f})"
)

# Render: coloured branch lines (binned by ratio) + downwind yz plane.
tips = start + axis * L[:, None]


def _layer_segments(mask: np.ndarray):
    if not mask.any():
        empty = np.empty(0, dtype=float)
        return empty, empty, empty
    s, t = start[mask], tips[mask]
    n = s.shape[0]
    x = np.empty(3 * n)
    y = np.empty(3 * n)
    z = np.empty(3 * n)
    x[0::3], x[1::3], x[2::3] = s[:, 0], t[:, 0], np.nan
    y[0::3], y[1::3], y[2::3] = s[:, 1], t[:, 1], np.nan
    z[0::3], z[1::3], z[2::3] = s[:, 2], t[:, 2], np.nan
    return x, y, z


RATIO_MAX = 1.0
fig = mt.figstyle.figure_3d(size="full")
fig.update_layout(
    scene=dict(aspectmode="data"),
    title=(
        f"U<sub>branch</sub> / U<sub>∞</sub> via momentum-wind CFD "
        f"(uniform inflow U = {K:.2f}, grid_size = {H_cell}, ν_diff = 0.03, storm +x)"
    ),
    showlegend=False,
)

# Downwind yz plane.
i_dw = cell_bounds_x.size - 2
y_centers = 0.5 * (cell_bounds_y[:-1] + cell_bounds_y[1:])
YY_plane, ZZ_plane = np.meshgrid(y_centers, z_centers_grid)
SC_plane = U_out[:, :, i_dw] / np.maximum(U_infty[:, None], 1e-6)
XX_plane = np.full_like(YY_plane, cell_bounds_x[-1])
fig.add_trace(
    go.Surface(
        x=XX_plane,
        y=YY_plane,
        z=ZZ_plane,
        surfacecolor=SC_plane,
        colorscale="Viridis",
        cmin=0.0,
        cmax=RATIO_MAX,
        showscale=False,
        opacity=0.85,
        contours={"x": {"show": False}, "y": {"show": False}, "z": {"show": False}},
    )
)

# Per-branch coloured traces, binned by ratio.
n_bins = 24
edges = np.linspace(0.0, RATIO_MAX, n_bins + 1)
bin_idx = np.clip(np.digitize(ratio, edges) - 1, 0, n_bins - 1)
centres = 0.5 * (edges[:-1] + edges[1:])
for b in range(n_bins):
    mask = bin_idx == b
    if not mask.any():
        continue
    x_seg, y_seg, z_seg = _layer_segments(mask)
    rgba = pcolors.sample_colorscale("Viridis", [float(np.clip(centres[b] / RATIO_MAX, 0.0, 1.0))])[
        0
    ]
    fig.add_trace(
        go.Scatter3d(
            x=x_seg,
            y=y_seg,
            z=z_seg,
            mode="lines",
            line=dict(color=rgba, width=3),
            hoverinfo="skip",
            showlegend=False,
        )
    )

# Invisible marker carrying the shared colourbar.
fig.add_trace(
    go.Scatter3d(
        x=[float("nan")],
        y=[float("nan")],
        z=[float("nan")],
        mode="markers",
        marker=dict(
            size=0.001,
            color=[0.0],
            colorscale="Viridis",
            cmin=0.0,
            cmax=RATIO_MAX,
            colorbar=dict(title="U / U<sub>∞</sub>"),
        ),
        showlegend=False,
        hoverinfo="skip",
    )
)
fig.show()

Magnitude is preserved across rotations (the canopy screening applies
regardless of direction); only the projection onto $(x, y)$ changes.
That's the rotation working.

## 4. Step 24 in pictures — growth under steady wind, then one big storm

There are two wind stages every MechaTree generation, and the Step-24
fixed-point loop only matters for the second:

1. **Sensing + storm pruning** (every generation). For the momentum
   model, sensing now sweeps `n_sensing_angles` *screened* directions
   (Step 26c) and the safety-factor genome sizes diameters from the
   worst-case stress; the per-generation storm wind from `wind_fn` is
   then fed to the Weibull failure law. Most gens cut a handful of
   branches; under a canopy-aware wind the pruning sub-loop iterates.
2. **Step 24 fixed-point** — the wind/prune loop only takes more than
   one pass when `wind_fn` is *canopy-aware* (3-arg). The default 2-arg
   sampler is a single pass; the momentum bridge from §3 iterates until
   the canopy stops moving.

To see both, we'll:

1. Grow a small forest with `wind_fn = unit_wind` — fixed amplitude
   $|U| = 1$, uniform-random direction every generation. Sensing and
   pruning both happen normally; the only change vs. the default is
   that we drop the Fortran exponential tail so the trees train
   against a *known* wind scale. Snapshot the canopy at three growth
   checkpoints to watch it fill out, each tree drawn in its own
   colour.
2. Hit the wind-sculpted canopy with one big storm at $U_\infty = 3$
   (~3× the sensing wind, so well above what the canopy was sized
   for). [`run_storm_replay`](../src/mechatree/wind/replay.py)
   snapshots every fixed-point iteration; we render one figure per
   iteration with surviving branches in their per-tree colour and
   branches cut **in this pass** overlaid in black. A red cone in
   each panel shows the storm direction.

What you'll see: iter 1 takes the largest cohort (branches sized for
$|U| = 1$ fail under the bigger storm); iter 2+ are smaller — partly
because the screened canopy-mean nudges up after each cut, partly
because the Weibull failure law re-rolls survivors at the new wind.
The loop exits when no more branches fall or the `eps_rel` test passes.

In [ ]:
# Sensing + storm pruning at fixed unit amplitude: wind_fn returns a
# (cos θ, sin θ, 0) vector every gen with θ uniform on [0, 2π). This
# is the legacy 2-arg storm path (single-pass per gen), so the
# Step-24 fixed-point loop is a no-op here — the trees just train
# against a steady |U| = 1 wind. We snapshot the canopy at three
# checkpoints to watch it fill out before the big storm.
# TREE_CFG (cell 1) pulls the branching angles from
# examples/forest.yaml.

from mechatree.wind.replay import StormPreSnapshot

warmup_cfg = mt.Config(
    tree=TREE_CFG,
    forest=mt.ForestConfig(size=8.0, n_trees_init=3, n_trees_max=12),
    n_generations=100,
)


def unit_wind(gen, rng):
    theta = rng.uniform(0.0, 2.0 * np.pi)
    return (np.cos(theta), np.sin(theta), 0.0)


warm = mt.Forest(warmup_cfg, seed=7, wind_fn=unit_wind)

checkpoints = [25, 50, 100]
growth_snapshots: list[tuple[int, StormPreSnapshot]] = []

g = 0
for target_gen in checkpoints:
    while g < target_gen:
        warm.step(g)
        g += 1
    per_tree = []
    for tree in warm.trees:
        start, axis, _D, L = tree.get_branch_data_batch()
        per_tree.append({"start": start, "axis": axis, "L": L})
    growth_snapshots.append((g, StormPreSnapshot(per_tree_geometry=per_tree)))
    n_b = sum(t.get_number_of_branches() for t in warm.trees)
    print(f"gen {g:>3}: {len(warm.trees):>2} trees, {n_b:>5} branches")


# ---------------------------------------------------------------------------
# Local rendering helpers — one 3D figure per call so panels stack
# vertically in the notebook output. Each tree gets a distinct hue
# (HSL cycle); branches cut in this pass overlay in black; the storm
# direction (when known) is drawn as a red cone above the canopy.
# ---------------------------------------------------------------------------


def _tree_segments(start, axis, L):
    n = start.shape[0]
    if n == 0:
        empty = np.empty(0, dtype=float)
        return empty, empty, empty
    tip = start + axis * L[:, None]
    x = np.empty(3 * n)
    y = np.empty(3 * n)
    z = np.empty(3 * n)
    x[0::3], x[1::3], x[2::3] = start[:, 0], tip[:, 0], np.nan
    y[0::3], y[1::3], y[2::3] = start[:, 1], tip[:, 1], np.nan
    z[0::3], z[1::3], z[2::3] = start[:, 2], tip[:, 2], np.nan
    return x, y, z


def _tree_palette(n: int) -> list[str]:
    return [f"hsl({int(360 * i / max(n, 1))}, 70%, 45%)" for i in range(n)]


def render_panel(pre_geometry, *, alive_masks=None, cut_masks=None, title="", storm_dir=None):
    fig = mt.figstyle.figure_3d(size="full")
    fig.update_layout(scene=dict(aspectmode="data"), title=title, showlegend=False)

    colors = _tree_palette(len(pre_geometry))

    # Surviving branches — one trace per tree so each gets its own colour.
    for i, geom in enumerate(pre_geometry):
        sel = slice(None) if alive_masks is None else alive_masks[i]
        s = geom["start"][sel]
        a = geom["axis"][sel]
        ell = geom["L"][sel]
        if s.shape[0] == 0:
            continue
        x, y, z = _tree_segments(s, a, ell)
        fig.add_trace(
            go.Scatter3d(
                x=x,
                y=y,
                z=z,
                mode="lines",
                line=dict(color=colors[i], width=2),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    # Branches cut this iteration — drawn black on top, slightly thicker.
    if cut_masks is not None:
        for i, geom in enumerate(pre_geometry):
            mask = cut_masks[i]
            if not mask.any():
                continue
            x, y, z = _tree_segments(geom["start"][mask], geom["axis"][mask], geom["L"][mask])
            fig.add_trace(
                go.Scatter3d(
                    x=x,
                    y=y,
                    z=z,
                    mode="lines",
                    line=dict(color="black", width=3),
                    hoverinfo="skip",
                    showlegend=False,
                )
            )

    # Storm direction arrow (single red cone above the canopy).
    if storm_dir is not None:
        all_xy = (
            np.concatenate([g["start"][:, :2] for g in pre_geometry])
            if pre_geometry
            else np.zeros((0, 2))
        )
        all_z = (
            np.concatenate([g["start"][:, 2] for g in pre_geometry])
            if pre_geometry
            else np.zeros(0)
        )
        if all_xy.size:
            cx, cy = float(all_xy[:, 0].mean()), float(all_xy[:, 1].mean())
            span = float(np.ptp(all_xy))
            z_top = float(all_z.max())
        else:
            cx, cy, span, z_top = 0.0, 0.0, 5.0, 5.0
        # Tail of arrow placed upwind from the canopy centre.
        tail_x = cx - storm_dir[0] * span * 0.6
        tail_y = cy - storm_dir[1] * span * 0.6
        z_arrow = z_top + 1.0
        fig.add_trace(
            go.Cone(
                x=[tail_x],
                y=[tail_y],
                z=[z_arrow],
                u=[storm_dir[0]],
                v=[storm_dir[1]],
                w=[0.0],
                sizemode="absolute",
                sizeref=span * 0.4,
                anchor="tail",
                showscale=False,
                colorscale=[[0, mt.figstyle.COLORS["red"]], [1, mt.figstyle.COLORS["red"]]],
                hoverinfo="skip",
            )
        )

    return fig


# Render the three growth checkpoints — all branches alive, no storm yet.
for gen, snap in growth_snapshots:
    n_b = sum(g["start"].shape[0] for g in snap.per_tree_geometry)
    render_panel(
        snap.per_tree_geometry,
        title=(
            f"growth under unit wind — gen {gen} "
            f"({len(snap.per_tree_geometry)} trees, {n_b} branches)"
        ),
    ).show()

In [ ]:
# One big storm, replayed pass-by-pass. With no angle_sampler the
# momentum bridge defaults to direction +x for every iteration, so the
# storm direction is stable across the replay. eps_rel=0 disables the
# canopy-mean early-exit so we see every wave the loop fires;
# max_iterations=5 caps it for the visual budget.
from mechatree.wind.momentum_wind import make_momentum_wind_fn

storm_wind = make_momentum_wind_fn(
    grid_size=2.0,
    pad_x=10.0,
    U_uniform=3.0,  # ~3× the unit sensing wind
)

pre_storm, snapshots = mt.run_storm_replay(
    warm,
    storm_wind,
    generation=g,
    max_iterations=5,
    eps_rel=0.0,
    leaf_drag_S0=warmup_cfg.tree.leaf_surface,
    cauchy=warmup_cfg.tree.cauchy,
)
for s in snapshots:
    print(
        f"iter {s.iteration}: cut {s.n_pruned_this_iter:>5} branches across "
        f"{s.n_trees_affected_this_iter:>2} trees "
        f"(cumulative {s.n_pruned_cumulative}), "
        f"wind=({s.wind_used[0]:+.3f}, {s.wind_used[1]:+.3f})"
    )

# Recover the storm direction from iter 1's wind vector (iter 0 is
# all-zero by construction). With angle_sampler=None this is +x.
wx, wy, _ = snapshots[1].wind_used if len(snapshots) > 1 else (1.0, 0.0, 0.0)
mag = float(np.hypot(wx, wy))
storm_dir = (wx / mag, wy / mag) if mag > 0 else (1.0, 0.0)

# One figure per iteration. Survivors keep their per-tree colour;
# branches cut between iter k-1 and iter k overlay in black so each
# panel shows *this pass*'s contribution, not the cumulative dead
# pile. The red cone marks the storm direction.
prev_alive = None
for s in snapshots:
    if s.iteration == 0:
        render_panel(
            pre_storm.per_tree_geometry,
            title="pre-storm (iter 0)",
            storm_dir=storm_dir,
        ).show()
    else:
        cut_now = [
            prev & ~curr for prev, curr in zip(prev_alive, s.alive_mask_per_tree, strict=True)
        ]
        render_panel(
            pre_storm.per_tree_geometry,
            alive_masks=s.alive_mask_per_tree,
            cut_masks=cut_now,
            title=(
                f"iter {s.iteration}: −{s.n_pruned_this_iter} branches on "
                f"{s.n_trees_affected_this_iter} trees "
                f"(cum −{s.n_pruned_cumulative})"
            ),
            storm_dir=storm_dir,
        ).show()
    prev_alive = s.alive_mask_per_tree

Reading the figures: each iteration's panel shows survivors in each
tree's own colour and *only this pass's* cut branches in black. The
red cone marks the storm direction (here +x, since
`angle_sampler=None`). Watch the black wave shift across iterations
— iter 1 takes the largest cohort (branches sized for $|U| = 1$
fail under the bigger storm); iters 2+ catch survivors as the
screened canopy-mean nudges up after each cut. The loop exits when no
branch falls or when `eps_rel` triggers; for clarity above we set
`eps_rel = 0` so every wave is rendered, even tiny ones.

Two caveats worth knowing:

- The wind shift between iterations is small in this regime: removing
  some branches barely moves the canopy-mean (it's an average over
  many occupied cells). The fixed-point loop still iterates because
  the Weibull failure law is stochastic — surviving branches get a
  fresh draw each pass at the slightly-elevated wind, and some that
  survived iter 1 lose the coin flip on iter 2. CLAUDE.md Step-24
  follow-up (b) (stress-floor cache) would suppress that re-roll
  artifact.
- Lineage colour is not stable across the *growth* checkpoints
  above, because trees can die / be born between checkpoints and the
  per-snapshot index reshuffles. Within the storm replay the index
  is frozen at pre-storm, so each tree keeps its colour from iter 0
  to iter 5.

## 5. Lab — prevailing wind via a von Mises CDF

Let's see what changes if the storm doesn't come uniformly from every
direction. A von Mises distribution concentrated around $\theta = 0$
models a 'prevailing easterly':

In [ ]:
# We'll just provide the CDF numerically since the von Mises CDF doesn't
# have a simple closed form; the Distribution class falls back to
# numerical inverse-CDF when SymPy can't invert symbolically.
kappa = 4.0
von_mises_cdf = f"(theta + (1/{kappa}) * sin({kappa} * (theta - pi))) / (2*pi)"
# (Not a real von Mises; just a heavy-on-0 CDF that's easy to write.)
prevailing = mt.Distribution(
    cdf_expr="theta/(2*pi) - sin(theta)/(2*pi)",
    var_name="theta",
    support=(0.0, 2 * math.pi),
)
samples = prevailing.sampler()(rng, 10_000)
fig = mt.figstyle.figure(size="full", aspect=9 / 4)
fig.add_trace(
    go.Histogram(
        x=samples,
        nbinsx=60,
        marker_color=mt.figstyle.COLORS["green"],
    )
)
fig.update_layout(
    title="Angle samples from a 'leans on 0' CDF",
    xaxis_title="θ (rad)",
    yaxis_title="count",
)
fig.show()

Plug a similar prevailing-wind CDF into the YAML, point the bridge at
the storm distribution, and run a long evolutionary tournament: you
should see trees with asymmetric canopies emerge — leaning *with*
the wind (the windward side gets pruned hard).

## What's next

The two big follow-ups flagged in earlier drafts have since landed:

1. **Screening-aware sensing (Step 26c)** — sensing no longer uses the
   legacy uniform 4-angle sweep. The `momentum` model now senses over
   `n_sensing_angles` *screened* directions (drawn from `angle_cdf`),
   so a branch reinforces against the same per-branch wind that prunes
   it. (The `default` model keeps the legacy unit-wind sweep.)
2. **Legacy bridges removed (Step 26)** — the `native` bulk-thinning
   and `dendroflow` bridges are gone; both collapsed the 3-D field to a
   scalar (a constant wind). The native momentum model is the sole
   canopy-aware path.

Remaining perf follow-up: a GIL-free Cython/OpenMP momentum kernel so
the `n_sensing_angles` independent solves can run in parallel (a
thread pool over the pure-NumPy kernel is GIL-bound). At island scale
the screening-aware tournament is already tractable serially
(~0.5 s/gen at R = 200; a 1000-gen run ≈ 9 min).